In [1]:
import os 
import numpy as np
data_dir = os.path.join("../dataset","1","data")
speakers_with_roi = ["s25_processed","s26_processed","s4_processed","s14_processed","s27_processed"]


In [2]:
import string 

chars = list(string.ascii_lowercase) + [' ']
blank_token = '-'
vocab = [blank_token] + chars

char2idx = {char: idx for idx, char in enumerate(vocab)}

In [ ]:
from torch.utils.data import Dataset, DataLoader 
from torch.nn.utils.rnn import pad_sequence 
from torch.utils.data import random_split


import torch

class LipReadingDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir 
        self.transform = transform
        self.video_files = []
        self.align_files = []
        for speaker in os.listdir(data_dir):
            if speaker in speakers_with_roi: #becuase of computational resources, most of the speakers have not been processed for lip roi extraction. So we will only use the speakers that have been processed for lip roi extraction
                speaker_dir = os.path.join(data_dir, speaker)
                for file in os.listdir(speaker_dir):
                    if file.endswith(".mpg"):
                        self.video_files.append(os.path.join(speaker_dir, file))
                        alignment_file = os.path.join(speaker_dir,'align',file.replace(".mpg", ".align"))
                        self.align_files.append(alignment_file)
                    
                    
       
    def __len__(self):
            return len(self.video_files)

    def __getitem__(self, idx):
            video_path = self.video_files[idx]
            align_path = self.align_files[idx]
            """sample align file 
                0 23750 sil
                23750 29500 bin
                29500 34000 blue
                34000 35500 at
                35500 41000 f
                41000 47250 two
                47250 53000 now
                53000 74500 sil"""
            roi_path = video_path.replace(".mpg", "_lip_roi.npy")
            roi_data = np.load(roi_path)
            if self.transform:
                roi_data = self.transform(roi_data)
            with open(align_path, 'r') as f:
                lines = f.readlines()
                ids= []
                for line in lines:
                    start, end, word = line.strip().split() 
                    chars = list(word.lower())
                    ids.extend([char2idx[char] for char in chars if char in char2idx])
                ids = np.array(ids, dtype=np.int32)
                return torch.tensor(roi_data, dtype=torch.float32), torch.tensor(ids, dtype=torch.int32)


dataset = LipReadingDataset(data_dir) 
train_size = int(0.8 * len(dataset)) 
val_size = len(dataset) - train_size 

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

def collate_fn(batch):
    roi_data, ids = zip(*batch)
    #roi data is already resized to 128 x 128 but ids can't be batched if they are of different lenghts
    roi_data = torch.stack(roi_data) 
    ids = pad_sequence(ids, batch_first=True, padding_value=0)
    return roi_data, ids 



train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)  
val_dataloader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn)      


(torch.Size([4, 75, 128, 128]), torch.Size([4, 28]))